# terra — Quickstart

Loads canned fixtures, shows both DataFrames, and prints a plan summary.

In [1]:
from pathlib import Path
import terra

FIXTURES = Path('../tests/fixtures')

## 1. Load state and inspect resources

In [2]:
state = terra.load.state_local(FIXTURES / 'state.json')
resources = terra.frame.resources_df(state)
resources

,address,module,type,name,provider,mode,schema_version,dependencies
0,aws_s3_bucket.assets,,aws_s3_bucket,assets,registry.terraform.io/hashicorp/aws,managed,0,[]
1,aws_iam_role.lambda_exec,,aws_iam_role,lambda_exec,registry.terraform.io/hashicorp/aws,managed,0,[]
2,module.networking.aws_vpc.main,module.networking,aws_vpc,main,registry.terraform.io/hashicorp/aws,managed,1,[]
3,module.networking.aws_subnet.public,module.networking,aws_subnet,public,registry.terraform.io/hashicorp/aws,managed,1,[module.networking.aws_vpc.main]


In [3]:
# Query just the S3 buckets at root level
resources.query("type == 'aws_s3_bucket' and module == ''")

,address,module,type,name,provider,mode,schema_version,dependencies
0,aws_s3_bucket.assets,,aws_s3_bucket,assets,registry.terraform.io/hashicorp/aws,managed,0,[]


## 2. Load a plan and inspect changes

In [4]:
plan = terra.load.plan_json(FIXTURES / 'plan.json')
terra.frame.summary(plan)

{'add': 1, 'change': 1, 'destroy': 1, 'no-op': 1}

In [5]:
changes = terra.frame.changes_df(plan)
changes[['address', 'type', 'actions', 'attr_diff']]

,address,type,actions,attr_diff
0,aws_s3_bucket.logs,aws_s3_bucket,[create],"[bucket, region]"
1,aws_iam_role.lambda_exec,aws_iam_role,[update],[tags]
2,aws_rds_cluster.db,aws_rds_cluster,[delete],"[cluster_identifier, engine]"
3,data.aws_caller_identity.current,aws_caller_identity,[no-op],[]


In [6]:
# Show only destructive changes
changes[changes['actions'].apply(lambda a: 'delete' in list(a))]

,address,module_address,type,name,provider,mode,actions,action_reason,attr_diff
2,aws_rds_cluster.db,,aws_rds_cluster,db,registry.terraform.io/hashicorp/aws,managed,[delete],,"[cluster_identifier, engine]"
